In [1]:
# 1) 필요한 라이브러리 임포트
import requests              # API HTTP 요청용
from lxml import etree       # XML 파싱용
import pandas as pd          # DataFrame 처리용

In [2]:
# API 키 / URL / 파라미터 설정
my_api_key = "cd398fb998a86593941ee42236c5c1bc0888f582bfb9c9cf453ce8d3c7ce5cdd"
api_url = "http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getCtprvnRltmMesureDnsty"

# API 기본 파라미터 (sidoName=전국으로 한 번에 전국 데이터 수집)
params = {
    "serviceKey": my_api_key,  # API 인증키
    "returnType": "xml",       # 응답 형식
    "numOfRows": "1000",       # 전국 측정소 약 700개, 여유있게 설정
    "pageNo": "1",             # 페이지 번호
    "sidoName": "전국",        # 전국
    "ver": "1.3",              # API 버전
}

In [3]:
# API 요청 - 최소 확인
response = requests.get(api_url, params=params)
print("성공 여부 200이면 성공 :", response.status_code)

성공 여부 200이면 성공 : 200


In [4]:
# XML 파싱 및 item 개수 확인

root = etree.fromstring(response.content)
print("성공여부 :", root.findtext('.//resultCode')) # 00이면 성공

items = root.findall('.//item')
print("item 개수:", len(items))

성공여부 : 00
item 개수: 672


In [5]:
# pandas.read_xml — XML 응답을 한 번에 DataFrame 으로 변환
df_readxml = pd.read_xml(response.content, xpath='//item')   # //item 경로의 모든 태그 추출
print('측정소 개수:', len(df_readxml))
df_readxml.head()

측정소 개수: 672


,pm25Grade1h,pm10Value24,so2Value,pm10Grade1h,pm10Value,o3Grade,pm25Flag,khaiGrade,pm25Value,no2Flag,...,sidoName,pm25Value24,no2Grade,o3Flag,pm25Grade,so2Flag,coGrade,dataTime,pm10Grade,o3Value
0,2.0,95,0.003,3.0,85,2.0,None,3.0,21,None,...,인천,21,1.0,None,2.0,None,1.0,2026-04-21 16:00,3.0,0.062
1,1.0,94,0.003,3.0,81,2.0,None,3.0,12,None,...,인천,12,1.0,None,1.0,None,1.0,2026-04-21 16:00,3.0,0.049
2,1.0,103,0.003,3.0,93,2.0,None,3.0,14,None,...,인천,15,1.0,None,1.0,None,1.0,2026-04-21 16:00,3.0,0.055
3,1.0,139,0.002,3.0,135,2.0,None,3.0,14,None,...,인천,17,1.0,None,2.0,None,1.0,2026-04-21 16:00,3.0,0.045
4,2.0,98,0.003,3.0,104,2.0,None,3.0,24,None,...,인천,24,1.0,None,2.0,None,1.0,2026-04-21 16:00,3.0,0.049


In [6]:
# 전국 데이터 한 번에 수집
response = requests.get(api_url, params=params)

root  = etree.fromstring(response.content)           # XML 바이트 → 트리 객체
items = root.findall('.//item')                      # <item> 태그 전부 찾기
print(f"전국 측정소 수: {len(items)}개")

# 필요한 5개 컬럼만 딕셔너리로 추출해 리스트에 누적
data_list = []
for item in items:
    data_list.append({
        'stationName': item.findtext('stationName'),   # 측정소 이름
        'dataTime':    item.findtext('dataTime'),      # 측정 시각
        'pm10Value':   item.findtext('pm10Value'),     # 미세먼지 (PM10)
        'pm25Value':   item.findtext('pm25Value'),     # 초미세먼지 (PM2.5)
        'sidoName':    item.findtext('sidoName'),      # 시/도 이름
    })

df = pd.DataFrame(data_list)                          # 리스트 → DataFrame
print(f"수집 완료: {len(df)}행")

전국 측정소 수: 672개
수집 완료: 672행


In [7]:
# 데이터 확인
print("shape:", df.shape)
print("시도별 측정소 수:")
print(df['sidoName'].value_counts())
print()
df.head()

shape: (672, 5)
시도별 측정소 수:
sidoName
경기    126
충남     54
경북     53
전남     50
경남     50
전북     48
인천     43
강원     40
서울     40
부산     36
충북     34
대구     26
울산     24
대전     15
제주     14
광주     13
세종      6
Name: count, dtype: int64



,stationName,dataTime,pm10Value,pm25Value,sidoName
0,길상,2026-04-21 16:00,85,21,인천
1,삼산,2026-04-21 16:00,81,12,인천
2,경인항,2026-04-21 16:00,93,14,인천
3,인천항,2026-04-21 16:00,135,14,인천
4,인천 신항,2026-04-21 16:00,104,24,인천


In [8]:
# CSV 저장 (utf-8-sig: 엑셀에서 한글 깨짐 방지)
df.to_csv('./sprint18_data.csv', index=False, encoding='utf-8-sig')
print('CSV 저장 완료!')

CSV 저장 완료!


In [9]:
from google.cloud import bigquery
from google.oauth2 import service_account

# BigQuery 연결 정보 설정
project_id   = 'project-87361850-33e3-4f44-b0f'  # GCP 프로젝트 ID
dataset_name = 'codeit'                            # BigQuery 데이터셋 (미리 생성 필요)
table_name   = 'sprint18_air_quality'              # 테이블명 (없으면 자동 생성)
key_file     = 'project-87361850-33e3-4f44-b0f-87d3cc5e279f.json'  # 서비스 계정 JSON 키 (같은 폴더)

# 서비스 계정 JSON 키로 인증 후 BigQuery 클라이언트 생성
credentials = service_account.Credentials.from_service_account_file(key_file)
bq_client = bigquery.Client(credentials=credentials, project=project_id)

# 적재 대상: 프로젝트.데이터셋.테이블
table_id = f'{project_id}.{dataset_name}.{table_name}'

job_config = bigquery.LoadJobConfig(
    autodetect=True,                          # 스키마 자동 감지
    source_format=bigquery.SourceFormat.CSV,  # CSV 형식
    skip_leading_rows=1,                      # 헤더 행 건너뜀
    write_disposition='WRITE_APPEND',         # 기존 데이터 유지하고 추가
)

# CSV 파일을 열어 변수에 담은 후 BigQuery에 업로드
csv_file = open('sprint18_data.csv', 'rb')
load_job = bq_client.load_table_from_file(csv_file, table_id, job_config=job_config)
load_job.result()  # 업로드 완료까지 대기

# 결과 확인
table = bq_client.get_table(table_id)
print(f'BigQuery 적재 완료: {table_id}')
print(f'총 행 수: {table.num_rows}')

BigQuery 적재 완료: project-87361850-33e3-4f44-b0f.codeit.sprint18_air_quality
총 행 수: 2016
